# Problem Statement

Large Language Models (LLMs) are typically trained on publicly available datasets and therefore lack access to **private or domain-specific documents**. As a result, when queried about information outside their training distribution, they may generate **inaccurate or hallucinated responses**.

To address this limitation, this project implements a **Retrieval-Augmented Generation (RAG)** pipeline using **LangChain**, enabling LLMs to dynamically access and reason over **external private knowledge sources**.

The system retrieves relevant context from a custom knowledge base and injects it into the LLM prompt, significantly improving accuracy and grounding responses in real data.

---

#  System Architecture: Question Answering Pipeline

The application follows a structured 3-stage pipeline:

##  1. Document Preparation (One-Time per Document)

This stage transforms raw documents into a searchable knowledge base:

- Load input documents (PDF, DOCX, TXT)
- Split documents into smaller, manageable chunks
- Convert text chunks into high-dimensional vector embeddings
- Store embeddings along with metadata in a **Vector Database**

---

##  2. Retrieval Phase (Per Query)

This stage identifies the most relevant information for a given user query:

- Convert the user query into an embedding vector
- Compute similarity scores between query embedding and stored document embeddings
- Rank document chunks based on semantic similarity
- Retrieve top-*k* most relevant chunks

---

##  3. Response Generation (Per Query)

This stage generates the final answer using contextual grounding:

- Combine the user query with retrieved document chunks
- Construct a structured prompt for the LLM
- Generate a context-aware response using the LLM
- Return the final answer to the user

---

#  Tech Stack (OPL Stack)

| Component        | Technology Used |
|----------------|----------------|
| **Vector Database** | Pinecone (scalable) / Chroma (lightweight) |
| **LLM Provider** | OpenAI API |
| **Framework** | LangChain |
| **Frontend/UI** | Streamlit |

---

# Key Features

-  **Multi-format Document Support**  
  Upload and process files in PDF, DOCX, and TXT formats

-  **Customizable Parameters**  
  - Adjustable **chunk size** for document splitting  
  - Configurable **top-k retrieval** for similarity search

-  **Conversational Memory**  
  Maintains chat history to enable context-aware interactions

-  **Dynamic Context Reset**  
  Automatically clears chat memory when:
  - A new document is uploaded  
  - Chunk size is modified  
  - Retrieval parameter (*k*) is changed  

- **Accurate, Context-Grounded Responses**  
  Eliminates hallucinations by anchoring answers in retrieved data

---

#  Core Concept: Retrieval-Augmented Generation (RAG)

This project leverages the RAG paradigm:

> Instead of relying solely on pre-trained knowledge, the model retrieves relevant external information at runtime and uses it to generate accurate, context-aware responses.

---

#  Use Cases

- Enterprise document search
- Research paper Q&A systems
- Internal knowledge base assistants
- Legal / medical document analysis
- Educational assistants for private study material


In [ ]:
# Load the env  
import os 
import langchain
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(),override=True)

In [ ]:
# Fucntion to load data from the pdf 
def load_document(file):
    try:
        from langchain_community.document_loaders import PyPDFLoader
        print(f'Loading file {file}.....')
        
        loader = PyPDFLoader(file)
        data = loader.load()
        
        return data
    
    except Exception as e:
        print(f"Error loading document: {e}")
        return None

In [ ]:
# Test it here 
data=load_document('paper.pdf')
print(data)

In [ ]:
print(data[0].page_content)

In [ ]:
print(data[1].metadata)

In [ ]:
# Display the number of pages in data 
print(f'The uploaded pdf has {len(data)} pages ')

# Display the number of characters in the page 
print(f'Page 1 has {len(data[0].page_content)} characters in it ')

In [ ]:
# Function that accepts other types of documents as well

def load_documnets(file):
    """
    This function accepsts files with extenstion '.txt' , '.pdf' and '.doxc'
    and returns data loaded from the file 

    """
    import os 
    name,extension=os.path.splitext(file)
    
    if extension == ".pdf":
        from langchain_community.document_loaders import PyPDFLoader
        print(f'Loading {file}....')
        loader=PyPDFLoader(file)

    elif extension == '.docx':
        from langchain_community.document_loaders import Docx2txtLoader
        print(f'Loading {file}....')
        loader=Docx2txtLoader(file)

    elif extension=='.txt':
        from langchain_community import TextLoader
        print(f'Loading the {file}......')
        loader=TextLoader(file)

    else:
        print('Document format is not supported for now ')
    
    data=loader.load()
    return data



In [ ]:
# Loading the file from the Public and Private Services 
# pip install wikipedia -q

def load_from_wikipedia(query,lang='en',load_max_docs=2):
    """
    This function will load data from the wikipedia

    query = Enter what you want to search in wikipedia
    lang  = Used for fetching response in particular language accepts en for english du for dutch etc

    """
    from langchain_community.document_loaders import WikipediaLoader

    loader=WikipediaLoader(query=query,lang=lang,load_max_docs=load_max_docs)

    data=loader.load()
    return data

In [ ]:
# pip install wikipedia

In [ ]:
import warnings 
warnings.filterwarnings("ignore")

In [ ]:
# Testing the wikipedia function
query="Islamic University of Science and Technology Awantipora"
data=load_from_wikipedia(query=query,lang='en')
print(data[0].page_content)

# Chunk the Data 

In [ ]:
# Function that is used for Chunking the data 
def chunk_data(data,chunk_size=256):
    """
    This function takes data loaded from lanchain_documentloaders and then chunks them 
    data : Data loaded using lanchain documents loader
    chunk_size : Provide an in integer ranging from 256 to any you like [256,512,1024 ]
    """
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=chunk_size)
    chunks=text_splitter.split_documents(data)
    return chunks

In [ ]:
# Load data using any document loader
data=load_from_wikipedia("Andrew NG")
# chunk it now 
chunks=chunk_data(data,chunk_size=256)
print(f'Obtaiend {len(chunks)} chunks of data ')

In [ ]:
print(chunks[0].page_content)

# Embed the Chunks and Upload them to a Vector Database

In [ ]:
# pip install torch torchvision torchaudio

In [ ]:
# pip install transformers==4.41.2

In [ ]:

# pip install sentence-transformers==2.7.0

In [24]:
def insert_or_fetch_embedding(index_name, chunks):
    import os
    import time
    from pinecone import Pinecone, ServerlessSpec
    from langchain_pinecone import PineconeVectorStore
    from langchain_community.embeddings import HuggingFaceEmbeddings
    from dotenv import load_dotenv, find_dotenv

    load_dotenv(find_dotenv(), override=True)

    # ✅ Pinecone init
    pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

    # ✅ LOCAL embeddings (NO API, NO ERRORS)
    embeddings = HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2"
    )

    existing_indexes = [i.name for i in pc.list_indexes()]

    if index_name in existing_indexes:
        print(f"✅ Index '{index_name}' exists → loading...")

        vector_store = PineconeVectorStore(
            index_name=index_name,
            embedding=embeddings
        )

    else:
        print(f"🚀 Creating index '{index_name}'...")

        pc.create_index(
            name=index_name,
            dimension=384,   # correct for MiniLM
            metric="cosine",
            spec=ServerlessSpec(
                cloud="aws",
                region="us-east-1"
            )
        )

        time.sleep(10)

        print("📦 Storing embeddings...")

        vector_store = PineconeVectorStore.from_documents(
            documents=chunks,
            embedding=embeddings,
            index_name=index_name
        )

        print("✅ Index created and data stored")

    return vector_store

In [ ]:
# pip install sentence-transformers

In [ ]:
# pip install sentence-transformers torch

In [25]:
# Test case
data=load_from_wikipedia("Something about Kashmir",lang='en')
chunks=chunk_data(data)
index_name='askwikipedia'
vector_store=insert_or_fetch_embedding(index_name=index_name,chunks=chunks)

🚀 Creating index 'askwikipedia'...
📦 Storing embeddings...
✅ Index created and data stored
